In [ ]:
# INSTALL LIBRARIES
!pip install datasets tensorflow scikit-learn nltk PyPDF2 gradio

In [ ]:
import pandas as pd
import re
import numpy as np
import gradio as gr

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Bidirectional,Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

import PyPDF2

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# LOAD DATASET

dataset = load_dataset("silentone0725/ai-human-text-detection-v1")
print(dataset)

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()


train_df.head()

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 36744
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 7874
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7874
    })
})


,text,label
0,This is going to be pretty similar to what Men...,human
1,"All life forms we know today , has the need of...",human
2,Hello and Welcome to ‘Ask A Doctor’ service. I...,human
3,Because light is massless . As an analogy for ...,human
4,We don't actually know what color dinosaurs li...,ai


In [ ]:
# Preprocessing

def clean_text(text):
    if not isinstance(text,str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

train_df['clean_text'] = train_df['text'].apply(clean_text)
val_df['clean_text'] = val_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

train_df['label'] = train_df['label'].map({'human':0,'ai':1})
val_df['label'] = val_df['label'].map({'human':0,'ai':1})
test_df['label'] = test_df['label'].map({'human':0,'ai':1})

In [ ]:
X_train = train_df['clean_text']
y_train = train_df['label']

X_val = val_df['clean_text']
y_val = val_df['label']

X_test = test_df['clean_text']
y_test = test_df['label']

In [ ]:
# EVALUATION FUNCTION

def evaluate_model(y_true, y_pred, model_name):
    print(f"\n{model_name} Evaluation")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
# ANN MODEL

vectorizer = TfidfVectorizer(max_features=3000)

X_train_tfidf = vectorizer.fit_transform(X_train).toarray()
X_val_tfidf = vectorizer.transform(X_val).toarray()
X_test_tfidf = vectorizer.transform(X_test).toarray()

model_ann = Sequential([
    Dense(128, activation='relu', input_shape=(3000,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training ANN...")
model_ann.fit(
    X_train_tfidf,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_tfidf, y_val)
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training ANN...
Epoch 1/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9222 - loss: 0.1883 - val_accuracy: 0.9488 - val_loss: 0.1312
Epoch 2/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9641 - loss: 0.0964 - val_accuracy: 0.9611 - val_loss: 0.1099
Epoch 3/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9841 - loss: 0.0462 - val_accuracy: 0.9674 - val_loss: 0.1089
Epoch 4/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9962 - loss: 0.0129 - val_accuracy: 0.9675 - val_loss: 0.1325
Epoch 5/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9994 - loss: 0.0032 - val_accuracy: 0.9682 - val_loss: 0.1523


In [ ]:
# Evaluate ANN
y_pred_ann = (model_ann.predict(X_test_tfidf) > 0.5).astype(int)
evaluate_model(y_test, y_pred_ann, "ANN")

247/247 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

ANN Evaluation
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      3937
           1       0.96      0.97      0.97      3937

    accuracy                           0.97      7874
   macro avg       0.97      0.97      0.97      7874
weighted avg       0.97      0.97      0.97      7874

Confusion Matrix:
[[3775  162]
 [ 103 3834]]


In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=3000)
tokenizer.fit_on_texts(X_train)

# Sequence Conversion
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding
max_len = 100
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

#  LSTM Model
model_lstm = Sequential([
    Embedding(input_dim=3000, output_dim=64, input_length=max_len),

    Bidirectional(LSTM(64,activation='tanh')),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# Compile
model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
print("Training LSTM...")
model_lstm.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_pad, y_val)
)

Training LSTM...
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1149/1149 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - accuracy: 0.9342 - loss: 0.1739 - val_accuracy: 0.9610 - val_loss: 0.1086
Epoch 2/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.9739 - loss: 0.0758 - val_accuracy: 0.9638 - val_loss: 0.0979
Epoch 3/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.9841 - loss: 0.0483 - val_accuracy: 0.9722 - val_loss: 0.0935
Epoch 4/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.9898 - loss: 0.0289 - val_accuracy: 0.9723 - val_loss: 0.1242
Epoch 5/5
1149/1149 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.9933 - loss: 0.0221 - val_accuracy: 0.9742 - val_loss: 0.1112


In [ ]:
# Evaluate LSTM
y_pred_lstm = (model_lstm.predict(X_test_pad) > 0.5).astype(int)
evaluate_model(y_test, y_pred_lstm, "LSTM")

247/247 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step

LSTM Evaluation
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      3937
           1       0.96      0.98      0.97      3937

    accuracy                           0.97      7874
   macro avg       0.97      0.97      0.97      7874
weighted avg       0.97      0.97      0.97      7874

Confusion Matrix:
[[3793  144]
 [  92 3845]]


In [ ]:
# PDF TEXT EXTRACTION
def extract_text(pdf_file):
    text = ""
    reader = PyPDF2.PdfReader(pdf_file)
    for page in reader.pages:
        text += page.extract_text() + " "
    return text.strip()

In [ ]:
# PREDICTION FUNCTIONS
def predict_ann(text):
    text = clean_text(text)
    text_tfidf = vectorizer.transform([text]).toarray()
    prob = model_ann.predict(text_tfidf)[0][0]
    return prob


In [ ]:
def predict_lstm(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    prob = model_lstm.predict(pad)[0][0]
    return prob

In [ ]:
def final_prediction(pdf_file):
    text = extract_text(pdf_file)

    ann_prob = predict_ann(text)
    lstm_prob = predict_lstm(text)

    # Probability Averaging
    final_prob = (ann_prob + lstm_prob) / 2

    # Final Decision
    if final_prob >= 0.5:
        return f"AI Generated\nConfidence: {final_prob*100:.2f}%"
    else:
        return f"Human Written\nConfidence: {(1-final_prob)*100:.2f}%"

In [ ]:
# GRADIO UI
interface = gr.Interface(
    fn=final_prediction,
    inputs=gr.File(label="Upload PDF"),
    outputs="text",
    title="AI vs Human Text Detector",
    description="Upload a PDF file to check if it is AI generated or human written."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7e171f2f9949be541.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
